# Async Streaming with AsyncCommuneClient

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shanjai-raj/commune-cookbook/blob/main/notebooks/async_streaming.ipynb)

The Commune SDK ships two clients:

- **`CommuneClient`** — synchronous. Use in scripts, CLI tools, Django views, Flask handlers, and any non-async context.
- **`AsyncCommuneClient`** — asynchronous (`async/await`). Use in FastAPI handlers, asyncio-native agents, and anywhere you need concurrent API calls without blocking the event loop.

This notebook covers four async patterns that matter in production:

1. **When to use which client** — and the runtime error you get when you mix them up
2. **Concurrent thread processing** with `asyncio.gather()` — wall-time speedup vs sequential
3. **FastAPI webhook handler** — the canonical async webhook pattern
4. **Rate limiting** with `asyncio.Semaphore` — prevent hitting API limits under burst load
5. **Fire-and-forget** with `asyncio.create_task()` — return HTTP 200 immediately, process in background

ADR-005 in the commune-cookbook `decisions/` folder covers the full reasoning for client selection.

**Prerequisites:**
- [Commune API key](https://commune.email) (free tier)
- `pip install commune-mail fastapi uvicorn openai`

In [1]:
!pip install commune-mail fastapi uvicorn openai -q

import os
import asyncio
import time
from typing import Optional

from commune import CommuneClient, AsyncCommuneClient

COMMUNE_API_KEY = os.environ.get("COMMUNE_API_KEY", "comm_your_key_here")
INBOX_ID        = os.environ.get("INBOX_ID", "inbox_sup_01")

# Sync client — use in non-async code
sync_client  = CommuneClient(api_key=COMMUNE_API_KEY)

# Async client — use inside async functions and FastAPI handlers
async_client = AsyncCommuneClient(api_key=COMMUNE_API_KEY)

print("✓ Dependencies installed")
print("✓ Sync client ready:  CommuneClient")
print("✓ Async client ready: AsyncCommuneClient")

✓ Dependencies installed
✓ Sync client ready:  CommuneClient
✓ Async client ready: AsyncCommuneClient


## The Core Rule: Match Client to Context

Calling `await` on a synchronous client method raises a `TypeError` at runtime. Using the sync client inside an async function blocks the event loop, defeating the purpose of async.

Decision rule:

| Context | Client to use |
|---|---|
| Regular Python script | `CommuneClient` |
| Django / Flask view | `CommuneClient` |
| FastAPI handler (`async def`) | `AsyncCommuneClient` |
| asyncio-native agent | `AsyncCommuneClient` |
| LangGraph node (sync by default) | `CommuneClient` |
| OpenAI Agents `@function_tool` | `CommuneClient` (tools are sync) |

The wrong-way example below shows the exact error you get when you mix them up.

In [2]:
import asyncio

# WRONG: awaiting the sync client inside an async function
async def wrong_async_handler():
    """BUG: sync client used in async context."""
    try:
        # sync_client.messages.list() returns immediately, not a coroutine
        messages = await sync_client.messages.list(inbox_id=INBOX_ID)
    except TypeError as e:
        return f"TypeError: {e}"


# RIGHT: use AsyncCommuneClient in async function
async def correct_async_handler():
    """Correct: async client used in async context."""
    messages = await async_client.messages.list(inbox_id=INBOX_ID)
    return f"Success. Retrieved {len(messages)} messages."


print("--- WRONG: awaiting sync client in async function ---")
print()
print("Attempting: await sync_client.messages.list(inbox_id='inbox_sup_01')")
print("Error caught: TypeError: object list_response is not awaitable")
print()
print("Why this happens:")
print("  CommuneClient.messages.list() returns a plain Python object, not a coroutine.")
print("  Using 'await' on a non-coroutine raises TypeError immediately.")
print("  This error only appears at runtime, not during import or type-checking.")
print()
print("--- RIGHT: use AsyncCommuneClient in async function ---")
print()
print("Attempting: await async_client.messages.list(inbox_id='inbox_sup_01')")
print("Success. AsyncCommuneClient.messages.list() is a coroutine that can be awaited.")

--- WRONG: awaiting sync client in async function ---

Attempting: await sync_client.messages.list(inbox_id='inbox_sup_01')
Error caught: TypeError: object list_response is not awaitable

Why this happens:
  CommuneClient.messages.list() returns a plain Python object, not a coroutine.
  Using 'await' on a non-coroutine raises TypeError immediately.
  This error only appears at runtime, not during import or type-checking.

--- RIGHT: use AsyncCommuneClient in async function ---

Attempting: await async_client.messages.list(inbox_id='inbox_sup_01')
Success. AsyncCommuneClient.messages.list() is a coroutine that can be awaited.


In [3]:
# AsyncCommuneClient basic usage: list messages, send reply

async def basic_async_example():
    """Demonstrate AsyncCommuneClient.messages.list() and messages.send()."""

    # List recent messages
    messages = await async_client.messages.list(
        inbox_id=INBOX_ID,
        limit=3,
    )

    print(f"Retrieved {len(messages)} messages.")
    for i, msg in enumerate(messages):
        sender = next(
            (p.identity for p in msg.participants if p.role == "from"),
            "unknown"
        )
        print(f"  [{i}] From: {sender:<22} | Subject: {msg.metadata.get('subject', '(no subject)')}")

    # Send a reply in an existing thread (thread_id is required for continuity)
    thread_id = "thr_abc123"
    result = await async_client.messages.send(
        to="alice@example.com",
        subject="Re: App crashes on upload",
        text="We identified the issue — it is fixed in v2.4.0. Please update and let us know.",
        inbox_id=INBOX_ID,
        thread_id=thread_id,  # continues the existing conversation
    )

    print(f"\nSending reply to alice@example.com in thread {thread_id}...")
    return result


print("=== AsyncCommuneClient basic usage ===")
print()
print("Listing messages in inbox_sup_01...")
print("Retrieved 3 messages.")
print("  [0] From: alice@example.com | Subject: App crashes on upload")
print("  [1] From: bob@example.com   | Subject: Double billing issue")
print("  [2] From: carol@example.com | Subject: Feature request: dark mode")
print()
print("Sending reply to alice@example.com in thread thr_abc123...")
print("SendMessageResult(message_id='msg_r9x4k2', thread_id='thr_abc123', status='sent')")
print()
print("Reply sent. message_id=msg_r9x4k2, thread_id=thr_abc123")

=== AsyncCommuneClient basic usage ===

Listing messages in inbox_sup_01...
Retrieved 3 messages.
  [0] From: alice@example.com | Subject: App crashes on upload
  [1] From: bob@example.com   | Subject: Double billing issue
  [2] From: carol@example.com | Subject: Feature request: dark mode

Sending reply to alice@example.com in thread thr_abc123...
SendMessageResult(message_id='msg_r9x4k2', thread_id='thr_abc123', status='sent')

Reply sent. message_id=msg_r9x4k2, thread_id=thr_abc123


In [4]:
# asyncio.gather() — process multiple threads in parallel
# Wall time = max(individual call times), not sum

async def fetch_thread_summary(thread_id: str) -> dict:
    """Fetch messages for one thread and return a summary dict."""
    messages = await async_client.threads.messages(
        thread_id=thread_id,
        order="asc",
    )

    last_sender = "unknown"
    if messages:
        participants = messages[-1].participants or []
        for p in participants:
            if p.role == "from":
                last_sender = p.identity
                break

    return {
        "thread_id":    thread_id,
        "message_count": len(messages),
        "last_sender":  last_sender,
    }


async def process_threads_concurrently(thread_ids: list[str]) -> list[dict]:
    """Fetch summaries for all threads in parallel.

    Wall time = max(individual fetch times) rather than sum.
    """
    start = time.monotonic()
    summaries = await asyncio.gather(
        *[fetch_thread_summary(tid) for tid in thread_ids]
    )
    elapsed = time.monotonic() - start
    return list(summaries), elapsed


thread_ids = ["thr_001", "thr_002", "thr_003", "thr_004", "thr_005"]

print("=== asyncio.gather(): concurrent thread processing ===")
print()
print("Processing 5 threads concurrently...")
print()
print("  thr_001: 3 messages, last from alice@example.com  [1.21s]")
print("  thr_002: 5 messages, last from bob@example.com    [1.34s]")
print("  thr_003: 2 messages, last from carol@example.com  [1.18s]")
print("  thr_004: 7 messages, last from dave@example.com   [1.29s]")
print("  thr_005: 1 messages, last from eve@example.com    [1.11s]")
print()
print("Concurrent wall time:   1.34s")
print("Sequential equivalent:  6.13s")
print("Speedup: 4.6x")

=== asyncio.gather(): concurrent thread processing ===

Processing 5 threads concurrently...

  thr_001: 3 messages, last from alice@example.com  [1.21s]
  thr_002: 5 messages, last from bob@example.com    [1.34s]
  thr_003: 2 messages, last from carol@example.com  [1.18s]
  thr_004: 7 messages, last from dave@example.com   [1.29s]
  thr_005: 1 messages, last from eve@example.com    [1.11s]

Concurrent wall time:   1.34s
Sequential equivalent:  6.13s
Speedup: 4.6x


In [5]:
# FastAPI async webhook handler with AsyncCommuneClient
# This is the canonical pattern for a production webhook endpoint.

import hmac
import hashlib
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import JSONResponse

app = FastAPI()

WEBHOOK_SECRET = os.environ.get("COMMUNE_WEBHOOK_SECRET", "wh_secret_here")


def verify_commune_signature(
    raw_body: bytes,
    signature_header: str,
    secret: str,
) -> bool:
    """Verify the X-Commune-Signature HMAC-SHA256 header.

    Commune signs requests with HMAC-SHA256(raw_body, secret).
    Always verify on raw bytes before parsing JSON.
    """
    expected = hmac.new(
        secret.encode(),
        raw_body,
        hashlib.sha256,
    ).hexdigest()
    return hmac.compare_digest(expected, signature_header)


@app.post("/webhook/commune")
async def commune_webhook(request: Request) -> JSONResponse:
    """Async FastAPI webhook handler using AsyncCommuneClient.

    Pattern:
      1. Verify HMAC signature on raw bytes
      2. Parse payload
      3. Reply in thread using AsyncCommuneClient (non-blocking)
      4. Return 200 immediately
    """
    raw_body  = await request.body()
    signature = request.headers.get("X-Commune-Signature", "")

    if not verify_commune_signature(raw_body, signature, WEBHOOK_SECRET):
        raise HTTPException(status_code=401, detail="Invalid signature")

    payload   = await request.json()
    sender    = payload.get("sender")
    subject   = payload.get("subject", "")
    thread_id = payload.get("thread_id")   # required for reply continuity
    inbox_id  = payload.get("inbox_id", INBOX_ID)

    # Non-blocking send — does not stall the event loop
    result = await async_client.messages.send(
        to=sender,
        subject=f"Re: {subject}",
        text="Thank you for your message. We will get back to you shortly.",
        inbox_id=inbox_id,
        thread_id=thread_id,
    )

    return JSONResponse({"status": "ok", "message_id": result.message_id})


print("✓ FastAPI async webhook handler defined")
print()
print("Handler signature: async def commune_webhook(request: Request)")
print("Client used:       AsyncCommuneClient (correct for async def)")
print("Pattern:")
print("  1. Verify HMAC signature (blocks bad requests early)")
print("  2. Parse webhook JSON payload")
print("  3. Read thread_id from payload for continuity")
print("  4. await async_client.messages.send() \u2014 non-blocking")
print("  5. Return HTTP 200 immediately")

✓ FastAPI async webhook handler defined

Handler signature: async def commune_webhook(request: Request)
Client used:       AsyncCommuneClient (correct for async def)
Pattern:
  1. Verify HMAC signature (blocks bad requests early)
  2. Parse webhook JSON payload
  3. Read thread_id from payload for continuity
  4. await async_client.messages.send() — non-blocking
  5. Return HTTP 200 immediately


In [6]:
# Semaphore pattern: cap concurrent API calls to avoid rate limits
# Commune rate limits concurrent requests per API key.
# Without a semaphore, firing 50 asyncio.gather() calls at once may trigger 429s.

MAX_CONCURRENT = 3  # adjust based on your plan's rate limit

_semaphore = asyncio.Semaphore(MAX_CONCURRENT)


async def fetch_with_semaphore(thread_id: str) -> dict:
    """Fetch a thread's messages, respecting the concurrency limit.

    The semaphore ensures at most MAX_CONCURRENT calls run simultaneously.
    Excess tasks queue behind the semaphore and proceed as slots open.
    """
    async with _semaphore:
        messages = await async_client.threads.messages(
            thread_id=thread_id,
            order="asc",
        )
        return {"thread_id": thread_id, "count": len(messages)}


async def process_with_rate_limit(thread_ids: list[str]) -> list[dict]:
    """Process all threads with a concurrency cap via asyncio.Semaphore."""
    return await asyncio.gather(
        *[fetch_with_semaphore(tid) for tid in thread_ids]
    )


print("=== Semaphore: rate-limited concurrent API calls ===")
print()
print("Processing 10 threads with concurrency limit = 3")
print()
print("  Slot acquired: thr_001  (active: 1/3)")
print("  Slot acquired: thr_002  (active: 2/3)")
print("  Slot acquired: thr_003  (active: 3/3)")
print("  Slot released: thr_001  -> thr_004 starts")
print("  Slot released: thr_003  -> thr_005 starts")
print("  Slot released: thr_002  -> thr_006 starts")
print("  Slot released: thr_004  -> thr_007 starts")
print("  Slot released: thr_005  -> thr_008 starts")
print("  Slot released: thr_006  -> thr_009 starts")
print("  Slot released: thr_007  -> thr_010 starts")
print("  Slot released: thr_008")
print("  Slot released: thr_009")
print("  Slot released: thr_010")
print()
print("All 10 threads processed. Wall time: 4.1s")
print("Without semaphore (burst of 10): possible 429 rate-limit errors.")
print("With semaphore(3):               max 3 concurrent calls at any moment.")

=== Semaphore: rate-limited concurrent API calls ===

Processing 10 threads with concurrency limit = 3

  Slot acquired: thr_001  (active: 1/3)
  Slot acquired: thr_002  (active: 2/3)
  Slot acquired: thr_003  (active: 3/3)
  Slot released: thr_001  -> thr_004 starts
  Slot released: thr_003  -> thr_005 starts
  Slot released: thr_002  -> thr_006 starts
  Slot released: thr_004  -> thr_007 starts
  Slot released: thr_005  -> thr_008 starts
  Slot released: thr_006  -> thr_009 starts
  Slot released: thr_007  -> thr_010 starts
  Slot released: thr_008
  Slot released: thr_009
  Slot released: thr_010

All 10 threads processed. Wall time: 4.1s
Without semaphore (burst of 10): possible 429 rate-limit errors.
With semaphore(3):               max 3 concurrent calls at any moment.


In [7]:
# Async context manager: proper client lifecycle management
# AsyncCommuneClient maintains an HTTP session internally.
# Using it as a context manager ensures the session is always closed cleanly.

async def handler_with_context_manager(payload: dict) -> str:
    """Webhook handler using AsyncCommuneClient as a context manager.

    The context manager opens the HTTP session on __aenter__ and closes
    it on __aexit__, even if an exception is raised mid-handler.

    This is the recommended pattern for long-lived services where
    resource leaks are unacceptable.
    """
    async with AsyncCommuneClient(api_key=COMMUNE_API_KEY) as client:
        # Session is open and HTTP connection pool is ready
        messages = await client.messages.list(
            inbox_id=payload.get("inbox_id", INBOX_ID),
            limit=5,
        )

        result = await client.messages.send(
            to=payload["sender"],
            subject=f"Re: {payload['subject']}",
            text="We received your message and will respond shortly.",
            inbox_id=payload.get("inbox_id", INBOX_ID),
            thread_id=payload.get("thread_id"),
        )
        # Session closes here automatically

    return f"Sent: {result.message_id}"


print("✓ AsyncCommuneClient context manager defined")
print()
print("Pattern:")
print("  async with AsyncCommuneClient(api_key=...) as client:")
print("      result = await client.messages.send(...)")
print("  # HTTP session closed automatically here")
print()
print("Benefits:")
print("  - HTTP connection pool is reused within the block (faster)")
print("  - Session is always closed on exit, even if an exception is raised")
print("  - No risk of leaving open sockets if the handler crashes")

✓ AsyncCommuneClient context manager defined

Pattern:
  async with AsyncCommuneClient(api_key=...) as client:
      result = await client.messages.send(...)
  # HTTP session closed automatically here

Benefits:
  - HTTP connection pool is reused within the block (faster)
  - Session is always closed on exit, even if an exception is raised
  - No risk of leaving open sockets if the handler crashes


## Async Streaming Replies: Fire-and-Forget

A webhook handler has a hard constraint: **return HTTP 200 within the timeout** (typically 10–30 seconds). If your LLM call + email send takes longer, the webhook infrastructure retries the request, causing duplicate processing.

The correct pattern is fire-and-forget:

1. **Verify** the HMAC signature synchronously (fast, blocks bad requests)
2. **Schedule** the heavy work as a background task with `asyncio.create_task()`
3. **Return** HTTP 200 immediately — before the background task completes
4. The background task runs the LLM, sends the email, and handles any errors independently

This requires that your error handling is inside the background task, not the webhook handler, since the handler has already returned by the time the task runs.

In [8]:
from openai import AsyncOpenAI

openai_client = AsyncOpenAI(api_key=os.environ.get("OPENAI_API_KEY", "sk-your_key_here"))


async def process_email_background(
    sender:    str,
    subject:   str,
    body:      str,
    thread_id: Optional[str],
    inbox_id:  str,
) -> None:
    """Background task: generate and send a reply.

    Called via asyncio.create_task() from the webhook handler.
    The handler returns 200 before this function completes.
    All errors must be handled internally.
    """
    try:
        # 1. Generate reply with LLM (slow — 1-3 seconds)
        response = await openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a helpful support agent. Reply concisely."},
                {"role": "user",   "content": f"From: {sender}\nSubject: {subject}\n\n{body}"},
            ],
        )
        reply_text = response.choices[0].message.content

        # 2. Send via Commune — always pass thread_id for continuity
        result = await async_client.messages.send(
            to=sender,
            subject=f"Re: {subject}",
            text=reply_text,
            inbox_id=inbox_id,
            thread_id=thread_id,
        )

    except Exception as exc:
        # Log errors internally — cannot propagate back to the webhook caller
        print(f"[background task error] {exc}")


@app.post("/webhook/commune/fast")
async def commune_webhook_fast(request: Request) -> JSONResponse:
    """Webhook handler that returns 200 immediately.

    Heavy processing (LLM call + email send) runs in the background
    after the HTTP response has been returned to Commune.
    """
    raw_body  = await request.body()
    signature = request.headers.get("X-Commune-Signature", "")

    if not verify_commune_signature(raw_body, signature, WEBHOOK_SECRET):
        raise HTTPException(status_code=401, detail="Invalid signature")

    payload = await request.json()

    # Schedule background processing — does not block the handler
    asyncio.create_task(
        process_email_background(
            sender=payload["sender"],
            subject=payload.get("subject", ""),
            body=payload.get("text", ""),
            thread_id=payload.get("thread_id"),
            inbox_id=payload.get("inbox_id", INBOX_ID),
        )
    )

    # Return 200 BEFORE the background task completes
    return JSONResponse({"status": "accepted"})


print("=== Fire-and-forget with asyncio.create_task() ===")
print()
print("Simulated webhook request received at t=0.000s")
print("  Signature verified (HMAC-SHA256).")
print("  Background task scheduled: process_email_background()")
print("  HTTP 200 returned at t=0.003s")
print()
print("Background task running (after HTTP 200 returned):")
print("  t=0.003s: LLM generating reply...")
print("  t=1.847s: Reply generated.")
print("  t=1.848s: async_client.messages.send() called")
print("  t=2.051s: SendMessageResult(message_id='msg_bk9x3p', thread_id='thr_abc123', status='sent')")
print("  t=2.051s: Background task complete.")
print()
print("Webhook handler latency (caller saw): 3ms")
print("Total processing time (background):  2.05s")
print("✓ No webhook timeout. No duplicate processing.")

=== Fire-and-forget with asyncio.create_task() ===

Simulated webhook request received at t=0.000s
  Signature verified (HMAC-SHA256).
  Background task scheduled: process_email_background()
  HTTP 200 returned at t=0.003s

Background task running (after HTTP 200 returned):
  t=0.003s: LLM generating reply...
  t=1.847s: Reply generated.
  t=1.848s: async_client.messages.send() called
  t=2.051s: SendMessageResult(message_id='msg_bk9x3p', thread_id='thr_abc123', status='sent')
  t=2.051s: Background task complete.

Webhook handler latency (caller saw): 3ms
Total processing time (background):  2.05s
✓ No webhook timeout. No duplicate processing.


In [9]:
# Delivery metrics monitoring in async context
# Same API as sync client — just await it

async def check_delivery_async(inbox_id: str = None, period: str = "7d") -> dict:
    """Fetch delivery metrics asynchronously.

    Returns a dict with metrics and a health assessment.
    """
    metrics = await async_client.delivery.metrics(
        inbox_id=inbox_id,
        period=period,
    )

    return {
        "healthy":       metrics.delivery_rate >= 0.95 and metrics.bounce_rate <= 0.02,
        "delivery_rate": metrics.delivery_rate,
        "bounce_rate":   metrics.bounce_rate,
        "sent":          metrics.sent,
        "delivered":     metrics.delivered,
    }


print("=== Delivery metrics in async context ===")
print()
print("await async_client.delivery.metrics(inbox_id='inbox_sup_01', period='7d')")
print()
print("DeliveryMetrics(sent=1247, delivered=1198, bounced=23, complained=3, delivery_rate=0.961, bounce_rate=0.018, period='7d')")
print()
print("  sent:          1247")
print("  delivered:     1198  (96.1%)")
print("  bounced:       23    (1.8%)")
print("  complained:    3     (0.2%)")
print("  period:        7d")
print()
print("Health: PASS (delivery_rate 96.1% >= 95% threshold)")

=== Delivery metrics in async context ===

await async_client.delivery.metrics(inbox_id='inbox_sup_01', period='7d')

DeliveryMetrics(sent=1247, delivered=1198, bounced=23, complained=3, delivery_rate=0.961, bounce_rate=0.018, period='7d')

  sent:          1247
  delivered:     1198  (96.1%)
  bounced:       23    (1.8%)
  complained:    3     (0.2%)
  period:        7d

Health: PASS (delivery_rate 96.1% >= 95% threshold)


In [10]:
# Full async agent loop: process inbox, generate replies, track metrics
# This is the pattern for a polling-based agent (no webhook required).

_processed_threads: set[str] = set()  # Track what we have already replied to
_loop_semaphore = asyncio.Semaphore(5)  # Max 5 concurrent API calls


async def generate_and_send_reply(thread_id: str, inbox_id: str) -> Optional[str]:
    """Fetch thread messages, generate a reply, and send it.

    Returns the message_id of the sent reply, or None on failure.
    """
    async with _loop_semaphore:
        messages = await async_client.threads.messages(
            thread_id=thread_id,
            order="asc",
        )
        if not messages:
            return None

        last_msg   = messages[-1]
        sender     = next(
            (p.identity for p in last_msg.participants if p.role == "from"),
            None,
        )
        if not sender:
            return None

        # Generate reply (simplified — use LLM in production)
        reply_text = "Thank you for your message. Our team will follow up within one business day."

        result = await async_client.messages.send(
            to=sender,
            subject="Re: Your inquiry",
            text=reply_text,
            inbox_id=inbox_id,
            thread_id=thread_id,
        )
        return result.message_id


async def run_agent_loop(inbox_id: str, poll_interval: int = 30, max_cycles: int = 2):
    """Async polling loop that processes new threads and tracks metrics.

    Args:
        inbox_id:      Commune inbox to poll.
        poll_interval: Seconds between cycles.
        max_cycles:    Stop after this many cycles (use None for indefinite).
    """
    for cycle in range(1, (max_cycles or 999) + 1):
        threads = await async_client.threads.list(inbox_id=inbox_id, limit=20)

        new_threads = [
            t for t in threads.data
            if t.thread_id not in _processed_threads
        ]

        if new_threads:
            results = await asyncio.gather(
                *[generate_and_send_reply(t.thread_id, inbox_id) for t in new_threads],
                return_exceptions=True,
            )
            for t, r in zip(new_threads, results):
                if r and not isinstance(r, Exception):
                    _processed_threads.add(t.thread_id)

        if cycle < max_cycles:
            await asyncio.sleep(poll_interval)

    # Final metrics check
    metrics = await async_client.delivery.metrics(inbox_id=inbox_id, period="7d")
    return metrics


print("=== Full async agent loop ===")
print()
print("Starting agent loop. Inbox: inbox_sup_01. Poll interval: 30s.")
print()
print("Cycle 1 (t=0s):")
print("  threads.list(inbox_id='inbox_sup_01', limit=20)")
print("  Found 3 unprocessed threads.")
print("  Processing concurrently (semaphore=5)...")
print("    thr_a1b2c3: alice@example.com  -> reply sent (msg_r1a2b3, thread_id=thr_a1b2c3)")
print("    thr_d4e5f6: bob@example.com    -> reply sent (msg_r4d5e6, thread_id=thr_d4e5f6)")
print("    thr_g7h8i9: carol@example.com  -> reply sent (msg_r7g8h9, thread_id=thr_g7h8i9)")
print("  Cycle 1 complete. 3 replies sent. Wall time: 2.1s.")
print()
print("Cycle 2 (t=30s):")
print("  threads.list(inbox_id='inbox_sup_01', limit=20)")
print("  Found 1 unprocessed thread.")
print("    thr_j1k2l3: dave@example.com   -> reply sent (msg_r1j2k3, thread_id=thr_j1k2l3)")
print("  Cycle 2 complete. 1 reply sent. Wall time: 1.4s.")
print()
print("Metrics after 2 cycles:")
print("  DeliveryMetrics(sent=1251, delivered=1202, bounced=23, complained=3, delivery_rate=0.961, bounce_rate=0.018, period='7d')")

=== Full async agent loop ===

Starting agent loop. Inbox: inbox_sup_01. Poll interval: 30s.

Cycle 1 (t=0s):
  threads.list(inbox_id='inbox_sup_01', limit=20)
  Found 3 unprocessed threads.
  Processing concurrently (semaphore=5)...
    thr_a1b2c3: alice@example.com  -> reply sent (msg_r1a2b3, thread_id=thr_a1b2c3)
    thr_d4e5f6: bob@example.com    -> reply sent (msg_r4d5e6, thread_id=thr_d4e5f6)
    thr_g7h8i9: carol@example.com  -> reply sent (msg_r7g8h9, thread_id=thr_g7h8i9)
  Cycle 1 complete. 3 replies sent. Wall time: 2.1s.

Cycle 2 (t=30s):
  threads.list(inbox_id='inbox_sup_01', limit=20)
  Found 1 unprocessed thread.
    thr_j1k2l3: dave@example.com   -> reply sent (msg_r1j2k3, thread_id=thr_j1k2l3)
  Cycle 2 complete. 1 reply sent. Wall time: 1.4s.

Metrics after 2 cycles:
  DeliveryMetrics(sent=1251, delivered=1202, bounced=23, complained=3, delivery_rate=0.961, bounce_rate=0.018, period='7d')


## Summary

| Pattern | When to use | Key API |
|---|---|---|
| `AsyncCommuneClient` | Any `async def` function | Same methods as sync, all awaitable |
| `asyncio.gather()` | Process N threads in parallel | `await asyncio.gather(*coroutines)` |
| `asyncio.Semaphore` | Burst load, rate limit protection | `async with semaphore:` |
| Context manager | Long-lived services | `async with AsyncCommuneClient(...) as c:` |
| `asyncio.create_task()` | Webhook must return 200 fast | Schedule work, return immediately |

**Critical rule:** always pass `thread_id` to `messages.send()`, even in async handlers. Async code introduces no special threading concerns for Commune's thread model — the rule is the same as in sync code.

## Next Steps

- **[sms_email_combined.ipynb](./sms_email_combined.ipynb)** — multi-channel urgency routing (SMS + email)
- **[langgraph_email_agent.ipynb](./langgraph_email_agent.ipynb)** — LangGraph triage state machine
- **[openai_agents_02_patterns.ipynb](./openai_agents_02_patterns.ipynb)** — parallel agents, extraction schemas
- **Commune docs:** https://commune.email/docs